# HW 1 — Quantum Kernel Classification

**CMU 17-620 | Fall 2026 | 100 points | Estimated 5–7 hours**

Extend Lab 2 by implementing two fixed quantum kernels, training classical SVMs on their matrices, comparing classical baselines, and estimating sampling uncertainty. Complete each TODO and the final analysis. Use the same train/test split and C=1.0 for every model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pennylane as qml
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity, rbf_kernel
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from sklearn.svm import SVC
print("PennyLane:",qml.__version__)

## A. Data and leakage (10 points)

We use two interleaving moons for a binary classification task. The following cell fixes one stratified training/test split and fits the scaler on **training inputs only**.

**Explain:** Why should we never fit preprocessing using held-out test examples? Why stratify?

In [ ]:
X,y=make_moons(n_samples=64,noise=0.13,random_state=17)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.25,stratify=y,random_state=17)
scaler=StandardScaler().fit(Xtr)
Ztr,Zte=scaler.transform(Xtr),scaler.transform(Xte)
assert Ztr.shape==(48,2) and Zte.shape==(16,2)
fig,axes=plt.subplots(1,2,figsize=(9,3.5))
for ax,p,l,title in [(axes[0],Xtr,ytr,"Training"),(axes[1],Xte,yte,"Test")]:
    ax.scatter(p[:,0],p[:,1],c=l,cmap="coolwarm",edgecolors="k")
    ax.set_title(title)
plt.show()

## B. Two encodings and overlap circuits (25 points)

**B1, 5 points:** For standardized 2D features z, map angle inputs to π tanh(z/2). Map amplitude inputs to the four-dimensional vector [z0, z1, 1, z0*z1]. The constant component prevents a zero vector.

**B2, 10 points:** Implement an inverse-circuit overlap using Y-angle embedding on two qubits.

**B3, 10 points:** Repeat using amplitude embedding of the four components and its adjoint. Both circuits return the probability of measuring |00⟩, not a signed inner product.

In [ ]:
def angle_features(z):
    # TODO B1: pi*tanh(z/2)
    raise NotImplementedError

def amplitude_features(z):
    # TODO B1: columns [z0,z1,1,z0*z1]
    raise NotImplementedError

# TODO: compute Ttr,Tte,Atr,Ate using the two transformations
# assert Ttr.shape==(48,2) and Atr.shape==(48,4)

In [ ]:
angle_dev=qml.device("default.qubit",wires=2)
amp_dev=qml.device("default.qubit",wires=2)

@qml.qnode(angle_dev)
def angle_overlap(x,y):
    # TODO B2: AngleEmbedding x, adjoint AngleEmbedding y, rotation="Y"
    return qml.expval(qml.Projector([0,0],wires=[0,1]))

@qml.qnode(amp_dev)
def amplitude_overlap(x,y):
    # TODO B3: AmplitudeEmbedding x, adjoint AmplitudeEmbedding y, normalize=True
    return qml.expval(qml.Projector([0,0],wires=[0,1]))

## C. Build and inspect quantum kernel matrices (20 points)

**C1 (10 points):** Write a function that computes one kernel entry for every pair of rows from two arrays. Build **48×48 training** and **16×48 test-versus-training** matrices for both quantum kernels.

**C2 (10 points):** Check the exact training matrices for symmetry, diagonal entries of 1, and positive semidefiniteness (within tolerance). Plot their heatmaps with an identical [0,1] color scale.

The test matrix must have **test rows and training columns** to work with a precomputed-kernel SVM.

In [ ]:
def kernel_matrix(left,right,kernel):
    # TODO C1
    raise NotImplementedError

# TODO C1: K_angle_tr,K_angle_te,K_amp_tr,K_amp_te
# TODO C2: check shape, symmetry, diagonals and eigenvalues
# TODO C2: plot side-by-side heatmaps, vmin=0,vmax=1

## D. Train SVMs and compare classical baselines (20 points)

Use four kernels on the **same train/test split**, with **C=1.0**:

1. Quantum angle encoding on the 2D angle features.
2. Quantum amplitude encoding on the lifted 4D features.
3. **Classical squared cosine** on the very same 4D lifted features.
4. **Classical RBF** on standardized original 2D features (gamma=0.5).

**D1 (10 points):** Build classical training/test matrices. Verify quantum amplitude equals squared classical cosine to numerical precision.

**D2 (10 points):** Fit four SVC models with precomputed kernels. Report accuracy, balanced accuracy, and confusion matrices on the same 16 held-out examples. Do not tune on held-out labels.

In [ ]:
# TODO D1: K_cos2_tr, K_cos2_te using cosine_similarity(Atr,... )**2
# TODO D1: K_rbf_tr,K_rbf_te using rbf_kernel(Ztr,...,gamma=0.5)
# assert np.allclose(K_amp_tr,K_cos2_tr,atol=1e-7)

def evaluate(Ktrain,Ktest,ytrain,ytest):
    # TODO D2: SVC(kernel="precomputed",C=1.0), fit, predict,
    # return predictions, accuracy, balanced accuracy, confusion matrix
    raise NotImplementedError

# TODO D2: evaluate and present ALL FOUR models

## E. How much do shots matter? (10 points)

**E1 (5 points):** Implement a sampled version of the angle overlap circuit that returns two-bit outcomes.

**E2 (5 points):** For one fixed input pair, estimate P(00) using **128** and **2048** shots via qml.set_shots. Compare both estimates with the exact overlap. Explain why more shots reduce uncertainty in expectation but do not guarantee a closer result in each experiment.

In [ ]:
sample_dev=qml.device("default.qubit",wires=2)
@qml.qnode(sample_dev)
def sampled_angle_overlap(x,y):
    # TODO E1: angle encoding of x and adjoint of y
    return qml.sample(wires=[0,1])

# TODO E2: choose one fixed pair Ttr[0],Ttr[1]
# sample_n=qml.set_shots(sampled_angle_overlap,shots=128)
# estimate=np.mean(np.all(sample_n(x,y)==0,axis=1))
# Repeat for 2048 shots and compare with angle_overlap(x,y)

## F. Conclusions (15 points)

Write **300–500 words**, using your own plots and measured results. Explain:

1. How angle and amplitude feature maps change the kernel geometry.
2. The four held-out accuracies, balanced accuracies, and at least one confusion matrix; note limitations of 16 test examples.
3. Why quantum amplitude overlap equals classical squared cosine **on the lifted four-feature inputs**. Does this establish quantum advantage?
4. What the SVM learned, and why no quantum gates were trained.
5. How the 128- and 2048-shot estimates differed, and what did **not** change.

Replace this paragraph with your analysis. Ensure every required code cell runs from a clean kernel.